In [1]:
library(numbat)
library(dplyr)
library(Seurat)
library(data.table)
library(future)
plan("multicore", workers = 12)
options(future.globals.maxSize = 10000 * 1024^10,
        future.rng.onMisuse = 'ignore')
sessionInfo()

Loading required package: Matrix


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, saveRDS


Loading Seurat v5 beta version 
To maintain compatibility with previous workflows, new Seurat objects will use the previous object structure by default
To use new Seurat v5 assays: Please run: options(Seurat.object.assay.version = 'v5')


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




R version 4.2.3 (2023-03-15)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 8.7 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /gpfs/home3/cruiz2/miniconda3/envs/r_python/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] future_1.32.0           data.table_1.14.8       Seurat_4.9.9.9040      
[4] SeuratObject_4.9.9.9081 sp_1.6-0                dplyr_1.1.1            
[7] numbat_1.2.3            Matrix_1.5-3           

loaded via a namespace (and not attached):
  [1] uuid_1.1-0             spam_

In [2]:
dmg <- readRDS('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds')
dmg

An object of class Seurat 
38576 features across 409561 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [4]:
dmg$numbat_id <- recode(dmg$predicted.annotation_level_3,
                         `AC-like` = 'Malignant',
                                       `MES-like` = "Malignant",    
                                       `OPC-like` = "Malignant", 
                                       `NPC-like` = "Malignant",    
                                       `Astrocyte` = "Glial_Neuronal", 
                                       # `Oligodendrocyte` = "Glial_Neuronal",    
                                       `OPC` = "Glial_Neuronal", 
                                       `Neuron` = "Glial_Neuronal",
                                        `Mono` = "Myeloid",
                                       `TAM-BDM` = "Myeloid",    
                                       `TAM-MG` = "Myeloid", 
                                       `DC` = "Myeloid",    
                                       `Mast` = "Myeloid",
                                       `CD4/CD8` = "Lymphoid",    
                                       `NK` = "Lymphoid", 
                                       `B cell` = "Lymphoid",    
                                       `Plasma B` = "Lymphoid",
                                       `Endothelial` = "Vascular",    
                                       `Mural cell` = "Vascular"
                         )
Idents(dmg) <- dmg$numbat_id
table(dmg@active.ident)


      Malignant  Glial_Neuronal        Vascular         Myeloid Oligodendrocyte 
         231886           78710            7188           60905           25955 
       Lymphoid 
           4917 

In [5]:
`%nin%` <- function(x, y) !(x %in% y)
dmg <- subset(dmg, Study %nin% c('Filbin2018', 'Liu2022'))
dmg
gc()

An object of class Seurat 
38576 features across 397794 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,8382606,447.7,16414178,876.7,16414178,876.7
Vcells,12653726388,96540.3,16213889217,123702.2,12897181288,98397.7


In [8]:
dmg <- subset(dmg, Study == 'Ruiz2023')
dmg

An object of class Seurat 
38576 features across 205805 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [9]:
ref_internal = aggregate_counts(GetAssayData(subset(dmg, 
                                                   idents = c('Malignant', 'Glial_Neuronal'),
                                                   invert = TRUE),
                                             slot = 'counts'
                                            ), 
                               as.data.frame(dmg@active.ident) %>% 
                                `colnames<-`('group') %>% 
                                tibble::rownames_to_column('cell') %>%
                                filter(!group %in% c('Malignant', 'Glial_Neuronal'))
                               )
ref_internal

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”


cell_dict
       Vascular         Myeloid Oligodendrocyte        Lymphoid 
           5538           35409           18407            3682 


,Vascular,Myeloid,Oligodendrocyte,Lymphoid
A1BG,6.210648e-06,6.937543e-06,1.800258e-05,1.096939e-05
A1CF,3.382036e-07,2.559213e-07,7.373789e-08,4.387757e-07
A2M,8.043712e-04,4.919084e-04,2.396482e-05,1.562041e-04
A2ML1,2.920850e-06,2.427794e-06,1.590632e-06,6.581635e-07
A3GALT2,1.199086e-06,3.057222e-06,7.479129e-07,8.775513e-07
A4GALT,3.609555e-05,3.596732e-07,4.318934e-07,3.290817e-07
A4GNT,3.074578e-08,1.245023e-07,5.266992e-08,1.096939e-07
AAAS,1.159116e-05,8.237900e-06,1.669637e-05,7.897962e-06
AACS,1.641825e-05,1.660722e-05,3.225506e-05,2.884950e-05
AADAC,2.767121e-07,2.628381e-07,6.320391e-08,1.096939e-07


### Process Ruiz2023

In [ ]:
ruiz <- SplitObject(dmg, split.by = 'SampleID')
ruiz
gc()

In [ ]:
table(dmg$SampleID)

In [ ]:
rm(dmg)
gc()

In [ ]:
#check cell name nomenclature to add to numbat pipeline
head(colnames(ruiz[[1]]))

In [15]:
length(ruiz)

[1] 51

In [16]:
for(i in 31:length(ruiz)) {  # run on the first half of ruiz
    cat(' ############################################\n',
        '### Numbat for dataset number ', names(ruiz[i]), '###\n',
        '############################################\n')
    
    old <- Sys.time() # get start time
    
    file_path <- paste0('/projects/0/einf2548/cruiz/cellranger_output/numbat_pileup_output_cellbender/',
                        names(ruiz[i]),'/',names(ruiz[i]),'_allele_counts.tsv.gz')
    
    if (file.exists(file_path)) {  # check if file exists
        df_allele = fread(file_path)
        df_allele$cell <- paste0(names(ruiz[i]),'_',df_allele$cell)
          
        # run
        out = run_numbat(
            as.matrix(GetAssayData(ruiz[[i]], slot = 'counts', assay = 'RNA')), # gene x cell integer UMI count matrix 
            ref_internal, # reference expression profile, a gene x cell type normalized expression level matrix
            df_allele, # allele dataframe generated by pileup_and_phase script and MUST BE A DATATABLE!
            min_cells = 20,
            t = 1e-3,
            max_iter = 2,
            init_k = 3,
            ncores = 40,
            ncores_nni = 40,
            multi_allelic = TRUE,
            plot = TRUE,
            genome = "hg38",
            out_dir = paste0('/projects/0/einf2548/cruiz/dmg/notebooks/iCNV/numbat/dmg_atlas_segregated_ref/Ruiz2023/',names(ruiz[i]))
        )
      
        cat(' ############################################\n',
        '### Done with dataset', names(ruiz[i]), '###\n',
        '############################################\n')
        
        # print elapsed time
        new <- Sys.time() - old # calculate difference
        print(new) # print in nice format
    } else {
        cat(' ############################################\n',
        '### Skipping dataset', names(ruiz[i]), 'because the file does not exist ###\n',
        '############################################\n')
    }
}


 ############################################
 ### Numbat for dataset number  T21-90581_212AAQ_region_2 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 273.9
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
913 cells

Mem used: 76.4Gb

Approximating initial clusters using smoothed expression ..

Mem used: 76.4Gb

number of genes left: 12505

running hclust...

Iteration 1

Mem used: 28.7Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.77). 

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

1 multi-allelic CNVs found: 15a

Evaluating CNV per cell ..

Mem used: 28.2Gb

All cells succeeded

Expanding allelic states..

Building phylogeny ..

Mem used: 28.2Gb

Using 5 CNVs to construct phylogeny

Using UPGMA tree as seed..

Mem used: 28.2

 ############################################
 ### Done with dataset T21-90581_212AAQ_region_2 ###
 ############################################
Time difference of 8.842005 mins
 ############################################
 ### Numbat for dataset number  T21-90610_232AAQ ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 1744.8
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
5816 cells

Mem used: 29.4Gb

Approximating initial clusters using smoothed expression ..

Mem used: 29.4Gb

number of genes left: 12592

running hclust...

Iteration 1

Mem used: 36.2Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.86). 

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

2 multi-allelic CNVs found: 14a,15c

Evaluating CNV per cell ..

Mem used: 31.8Gb

All cells succeeded

Expanding allelic states..

Building phylogeny ..

Mem used: 31.9Gb

Using 5 CNVs to construct phylogeny

Using UPGMA tree as seed..

Mem used

 ############################################
 ### Done with dataset T21-90610_232AAQ ###
 ############################################
Time difference of 48.63547 mins
 ############################################
 ### Numbat for dataset number  T21-90717_MIMIC002 ###
 ############################################


...

 ############################################
 ### Done with dataset VUMC11_thalamus ###
 ############################################
Time difference of 57.72861 mins
 ############################################
 ### Numbat for dataset number  VUMC17_healthy ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 1458.9
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
4863 cells

Mem used: 30.2Gb

Approximating initial clusters using smoothed expression ..

Mem used: 30.2Gb

number of genes left: 12818

running hclust...

Iteration 1

Mem used: 35.1Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.65). 

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 32.1Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phyl

 ############################################
 ### Done with dataset VUMC17_healthy ###
 ############################################
Time difference of 42.14408 mins
 ############################################
 ### Numbat for dataset number  VUMC17 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 1537.8
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
5126 cells

Mem used: 29.6Gb

Approximating initial clusters using smoothed expression ..

Mem used: 29.6Gb

number of genes left: 12345

running hclust...

Iteration 1

Mem used: 34.7Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.82). 

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 30.3Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phyl

 ############################################
 ### Done with dataset VUMC17 ###
 ############################################
Time difference of 40.37678 mins


### Samples that did not detect CNVs with default settings

In [14]:
which(names(ruiz) == 'T21-90813_MIMIC006')
which(names(ruiz) == 'T21-91022_MIMIC008')
which(names(ruiz) == 'VU314')

[1] 35

[1] 37

[1] 46

In [14]:
for(i in c(35, 37, 46)) {  # run on the first half of ruiz
    cat(' ############################################\n',
        '### Numbat for dataset number ', names(ruiz[i]), '###\n',
        '############################################\n')
    
    old <- Sys.time() # get start time
    
    file_path <- paste0('/projects/0/einf2548/cruiz/cellranger_output/numbat_pileup_output_cellbender/',
                        names(ruiz[i]),'/',names(ruiz[i]),'_allele_counts.tsv.gz')
    
    if (file.exists(file_path)) {  # check if file exists
        df_allele = fread(file_path)
        df_allele$cell <- paste0(names(ruiz[i]),'_',df_allele$cell)
          
        # run
        out = run_numbat(
            as.matrix(GetAssayData(ruiz[[i]], slot = 'counts', assay = 'RNA')), # gene x cell integer UMI count matrix 
            ref_internal, # reference expression profile, a gene x cell type normalized expression level matrix
            df_allele, # allele dataframe generated by pileup_and_phase script and MUST BE A DATATABLE!
            min_cells = 20,
            t = 1e-3,
            max_iter = 2,
            init_k = 5, # changed this parameter 
            max_entropy = 0.8, # changed this parameter
            ncores = 32,
            ncores_nni = 32,
            multi_allelic = TRUE,
            plot = TRUE,
            genome = "hg38",
            out_dir = paste0('/projects/0/einf2548/cruiz/dmg/notebooks/iCNV/numbat/dmg_atlas_segregated_ref/Ruiz2023/',names(ruiz[i]))
        )
      
        cat(' ############################################\n',
        '### Done with dataset', names(ruiz[i]), '###\n',
        '############################################\n')
        
        # print elapsed time
        new <- Sys.time() - old # calculate difference
        print(new) # print in nice format
    } else {
        cat(' ############################################\n',
        '### Skipping dataset', names(ruiz[i]), 'because the file does not exist ###\n',
        '############################################\n')
    }
}

Iteration 1

Mem used: 29.9Gb

Running HMMs on 9 cell groups..

Expression noise level: medium (0.53). 

Running HMMs on 5 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 28.9Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phylogeny ..

Mem used: 28.9Gb

No CNV remains after filtering by entropy in single cells. Consider increasing max_entropy.



 ############################################
 ### Done with dataset T21-90813_MIMIC006 ###
 ############################################
Time difference of 7.47044 mins
 ############################################
 ### Numbat for dataset number  T21-91022_MIMIC008 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 5
max_cost = 1831.8
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.8
skip_nj = FALSE
diploid_chroms = 
ncores = 32
ncores_nni = 32
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
6106 cells

Mem used: 29.5Gb

Approximating initial clusters using smoothed expression ..

Mem used: 29.5Gb

number of genes left: 12348

running hclust...

Iteration 1

Mem used: 36.6Gb

Running HMMs on 9 cell groups..

Expression noise level: low (0.47). 

Running HMMs on 5 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 33Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phylogeny

 ############################################
 ### Done with dataset T21-91022_MIMIC008 ###
 ############################################
Time difference of 53.75608 mins
 ############################################
 ### Numbat for dataset number  VU314 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 5
max_cost = 1239.9
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.8
skip_nj = FALSE
diploid_chroms = 
ncores = 32
ncores_nni = 32
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
4133 cells

Mem used: 30Gb

Approximating initial clusters using smoothed expression ..

Mem used: 30Gb

number of genes left: 12488

running hclust...

Iteration 1

Mem used: 33.5Gb

Running HMMs on 9 cell groups..

Expression noise level: medium (0.69). 

Running HMMs on 5 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 31.4Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phylogen

 ############################################
 ### Done with dataset VU314 ###
 ############################################
Time difference of 17.43709 mins


In [15]:
for(i in c(35, 46)) {  # run on the first half of ruiz
    cat(' ############################################\n',
        '### Numbat for dataset number ', names(ruiz[i]), '###\n',
        '############################################\n')
    
    old <- Sys.time() # get start time
    
    file_path <- paste0('/projects/0/einf2548/cruiz/cellranger_output/numbat_pileup_output_cellbender/',
                        names(ruiz[i]),'/',names(ruiz[i]),'_allele_counts.tsv.gz')
    
    if (file.exists(file_path)) {  # check if file exists
        df_allele = fread(file_path)
        df_allele$cell <- paste0(names(ruiz[i]),'_',df_allele$cell)
          
        # run
        out = run_numbat(
            as.matrix(GetAssayData(ruiz[[i]], slot = 'counts', assay = 'RNA')), # gene x cell integer UMI count matrix 
            ref_internal, # reference expression profile, a gene x cell type normalized expression level matrix
            df_allele, # allele dataframe generated by pileup_and_phase script and MUST BE A DATATABLE!
            min_cells = 20,
            t = 1e-3,
            max_iter = 2,
            init_k = 10, # changed this parameter 
            max_entropy = 1, # changed this parameter
            ncores = 32,
            ncores_nni = 32,
            multi_allelic = TRUE,
            plot = TRUE,
            genome = "hg38",
            out_dir = paste0('/projects/0/einf2548/cruiz/dmg/notebooks/iCNV/numbat/dmg_atlas_segregated_ref/Ruiz2023/',names(ruiz[i]))
        )
      
        cat(' ############################################\n',
        '### Done with dataset', names(ruiz[i]), '###\n',
        '############################################\n')
        
        # print elapsed time
        new <- Sys.time() - old # calculate difference
        print(new) # print in nice format
    } else {
        cat(' ############################################\n',
        '### Skipping dataset', names(ruiz[i]), 'because the file does not exist ###\n',
        '############################################\n')
    }
}

 ############################################
 ### Numbat for dataset number  T21-90813_MIMIC006 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 10
max_cost = 535.5
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 1
skip_nj = FALSE
diploid_chroms = 
ncores = 32
ncores_nni = 32
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
1785 cells

Mem used: 29Gb

Approximating initial clusters using smoothed expression ..

Mem used: 29Gb

number of genes left: 12405

running hclust...

Iteration 1

Mem used: 29.9Gb

Running HMMs on 18 cell groups..

Expression noise level: low (0.48). 

Running HMMs on 9 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 29.4Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phylogeny ..

 ############################################
 ### Done with dataset T21-90813_MIMIC006 ###
 ############################################
Time difference of 15.88219 mins
 ############################################
 ### Numbat for dataset number  VU314 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 10
max_cost = 1239.9
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 1
skip_nj = FALSE
diploid_chroms = 
ncores = 32
ncores_nni = 32
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
4133 cells

Mem used: 28.7Gb

Approximating initial clusters using smoothed expression ..

Mem used: 28.7Gb

number of genes left: 12488

running hclust...

Iteration 1

Mem used: 33.5Gb

Running HMMs on 19 cell groups..

Expression noise level: medium (0.65). 

Running HMMs on 10 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 32.4Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phy

 ############################################
 ### Done with dataset VU314 ###
 ############################################
Time difference of 36.56692 mins
